# Stack Compass — Análisis narrado

**Pregunta:** ¿Qué stacks tecnológicos maximizan el trinomio salario / satisfacción / empleabilidad para un perfil data-IA junior en la UE?

**Fuente:** Stack Overflow Annual Developer Survey (año más reciente disponible).  
**Población:** Roles data/IA (núcleo + frontera) en los 27 países de la UE.

In [1]:
import sys
sys.path.insert(0, "../src")

from load import load_raw, get_duckdb_connection, load_processed
from clean import filter_population, clean_salary, filter_geography, add_region
from metrics import salary_by_stack, satisfaction_by_stack, employability_by_stack

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Carga de datos

Leemos el CSV crudo sin inferencia de tipos (`low_memory=False`) para evitar advertencias en columnas mixtas.

In [2]:
df_raw = load_raw()
print(f"Filas: {len(df_raw):,}  |  Columnas: {df_raw.shape[1]}")
df_raw.head(2)

Filas: 49,191  |  Columnas: 172


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0


## 2. Limpieza y filtrado de población

Nos quedamos con respondentes en la UE que declaran roles data/IA.  
Después saneamos el salario: los valores fuera del rango 5.000–300.000 USD se marcan como NaN (umbrales fijos, no percentiles). No se eliminan filas, solo se anula el salario corrupto.

In [ ]:
pob = filter_population(df_raw)
pob = filter_geography(pob)
pob = add_region(pob)
print(f"Población filtrada: {len(pob):,} filas")

pob_sal = clean_salary(pob)
validos = pob_sal["ConvertedCompYearly"].notna().sum()
print(f"Salarios marcados como NaN: {pob_sal['ConvertedCompYearly'].isna().sum()}")
print(f"Con salario válido (5k–300k USD): {validos:,}")

pob_sal.to_csv("../data/processed/survey_clean.csv", index=False)
print("CSV limpio guardado en data/processed/survey_clean.csv")

Población filtrada: 1,194 filas
NaN en salario: 297
Con salario válido (p1-p99): 1,194 filas


In [4]:
pob_sal.to_csv("../data/processed/survey_clean.csv", index=False)
print("CSV limpio guardado en data/processed/survey_clean.csv")

CSV limpio guardado en data/processed/survey_clean.csv


## 3. Métricas del trinomio

Calculamos las tres dimensiones de interés sobre la misma población.

### 3.1 Salario por stack

Mediana de `ConvertedCompYearly` (USD/año) agrupada por tecnología principal.

In [5]:
df_sal = salary_by_stack(pob_sal)
df_sal.head(10)

,Region,tech,median_salary,p25,p75,n
0,España,TypeScript,73669.0,58007.0,87011.0,17
1,España,Bash/Shell (all shells),69609.0,45043.0,84691.0,35
2,España,Python,69029.0,46449.5,84111.0,56
3,España,SQL,65548.0,48871.0,81210.0,38
4,España,HTML/CSS,63808.0,41765.0,81210.0,25
5,España,JavaScript,58007.0,35268.5,81210.0,23
6,UE sin España,MicroPython,87011.0,53366.5,104413.0,15
7,UE sin España,Groovy,83531.0,58007.0,109866.0,69
8,UE sin España,MATLAB,81088.0,54817.0,102188.5,26
9,UE sin España,TypeScript,80050.0,55687.0,108621.5,230


### 3.2 Satisfacción por stack

Satisfacción = retención (Admired). Para cada tech, qué fracción de quienes la usaron también la admira (LanguageAdmired = usó y repetiría). Score 0–100%.

In [6]:
df_sat = satisfaction_by_stack(pob)
df_sat.head(10)

,Region,tech,n,admired_pct
0,España,Python,66,56.060606
1,España,TypeScript,20,55.000000
2,España,Bash/Shell (all shells),40,50.000000
3,España,SQL,45,46.666667
4,España,HTML/CSS,30,26.666667
5,España,JavaScript,27,25.925926
6,UE sin España,GDScript,28,82.142857
7,UE sin España,Zig,19,78.947368
8,UE sin España,Rust,172,70.348837
9,UE sin España,Python,860,62.441860


### 3.3 Empleabilidad por stack

Empleabilidad = % de uso. Porcentaje de respondentes que declaran cada tech, sobre quienes contestaron la pregunta de stack en su región. Comparable entre España y UE pese a la diferencia de muestra.

In [7]:
df_emp = employability_by_stack(pob)
df_emp.head(20)

,tech,Region,n,pct,rank
0,Python,España,66,91.666667,1
1,SQL,España,45,62.500000,2
2,Bash/Shell (all shells),España,40,55.555556,3
3,HTML/CSS,España,30,41.666667,4
4,JavaScript,España,27,37.500000,5
5,TypeScript,España,20,27.777778,6
6,Python,UE sin España,860,84.066471,1
7,Bash/Shell (all shells),UE sin España,648,63.343109,2
8,SQL,UE sin España,628,61.388074,3
9,JavaScript,UE sin España,459,44.868035,4


## 4. Visualización


In [ ]:
from plots import build_dashboard

fig = build_dashboard(df_sal, df_sat, df_emp)
plt.show()

ModuleNotFoundError: No module named 'src'

## 5. Conclusiones

![Dashboard España vs UE](figures/dashboard.png)

- **Empleabilidad — Python es casi monocultivo en España.** Lo usa ~92% del perfil data/IA español, frente a ~83% en la UE. La diferencia real es la dispersión: la UE reparte el uso entre muchas más tecnologías (Go, Rust, C#, Java…); el stack data español se concentra en un puñado.
- **Satisfacción — el contraste más claro.** En España, Python y TypeScript lideran la retención (~55%), pero HTML/CSS y JavaScript caen a ~26%: la mitad de quien los usa no los repetiría. En la UE el top lo copan techs emergentes (GDScript, Zig, Rust: 70–82%) sin muestra suficiente en España. De las techs comunes, Python gana en satisfacción en ambas regiones.
- **Salario — banda estrecha y muy incierta en España.** ⚠️ *(mete aquí las medianas exactas de la tabla: rango ~__k–__k USD, TypeScript arriba, JavaScript abajo).* Las barras de error p25–p75 son casi tan anchas como las barras: con 70 personas con salario, el ranking salarial español es orientativo. La UE paga más en todo y tiene más techs bien pagadas.
- **El hallazgo que enmarca todo:** España supera el umbral n≥15 en solo 6 tecnologías por métrica; la UE en 15. No es una carencia del análisis — es el retrato de un mercado más pequeño donde pocas tecnologías alcanzan masa crítica. La comparación se juega en esas 6 comunes (Python, SQL, Bash/Shell, JavaScript, HTML/CSS, TypeScript).

